In [0]:
import time
from datetime import datetime

notebook_start = time.time()

def log(message, level="INFO"):
    """Print a timestamped log line so it is visible in notebook and job run output."""
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] [{level}] {message}")

dbutils.widgets.text("table_name","")
dbutils.widgets.text("landing_timestamp","")
dbutils.widgets.text("merge_keys","")
dbutils.widgets.text("load_type","")

table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

log("========== Bronze_to_Silver started ==========")
log(f"Parameter table_name        = '{table_name}'")
log(f"Parameter landing_timestamp = '{landing_timestamp}'")
log(f"Parameter merge_keys        = '{merge_keys}'")
log(f"Parameter load_type         = '{load_type}'")

for param_name, param_value in {"table_name": table_name, "landing_timestamp": landing_timestamp}.items():
    if not param_value.strip():
        log(f"Parameter '{param_name}' is empty", "WARN")

In [0]:
source_table = f"adb_caresync.bronze.{table_name}"
step_start = time.time()
log(f"Step 1: Reading {source_table} where landing_timestamp='{landing_timestamp}'")

try:
    bronze_df = spark.read.table(source_table).filter(
        f"landing_timestamp='{landing_timestamp}'"
    )
    bronze_count = bronze_df.count()
except Exception as e:
    log(f"Failed to read {source_table}: {e}", "ERROR")
    raise

log(f"Rows read from bronze: {bronze_count}")
log(f"Source columns ({len(bronze_df.columns)}): {bronze_df.columns}")

if bronze_count == 0:
    log("No rows matched this landing_timestamp. Latest values available in bronze:", "WARN")
    spark.read.table(source_table).select("landing_timestamp").distinct() \
        .orderBy("landing_timestamp", ascending=False).limit(5).show(truncate=False)

log(f"Step 1 completed in {time.time() - step_start:.2f}s")


In [0]:
from pyspark.sql import functions as F

target_table = f"adb_caresync.silver.{table_name}"
normalised_load_type = (load_type or 'FULL').strip().upper()
merge_key_list = [key.strip() for key in merge_keys.split(',') if key.strip()]

step_start = time.time()
log("Step 2: Validating configuration and adding audit columns")
log(f"Target table     : {target_table}")
log(f"Load type        : '{load_type}' -> normalised to '{normalised_load_type}'")
log(f"Merge key list   : {merge_key_list}")

if normalised_load_type=='MERGE':
    if len(merge_key_list)==0:
        log("load_type is MERGE but no merge_keys were provided", "ERROR")
        raise ValueError(f"Merge keys need to be provided when load_type is 'MERGE'")
    missing_merge_keys = [key for key in merge_key_list if key not in bronze_df.columns]
    if len(missing_merge_keys)>0:
        log(f"Merge keys missing from source: {missing_merge_keys}. Available columns: {bronze_df.columns}", "ERROR")
        raise ValueError(f"Merge keys not found in source data: {missing_merge_keys}")

updated_df = bronze_df.withColumn("updated_timestamp",F.current_timestamp()).withColumn("insert_timestamp",F.current_timestamp())

if normalised_load_type=='MERGE':
    duplicate_keys = bronze_df.groupBy(*merge_key_list).count().filter("count > 1").count()
    log(f"Duplicate merge key combinations in source: {duplicate_keys}")
    if duplicate_keys > 0:
        log("Source has duplicate merge keys; MERGE will fail if multiple source rows match one target row", "WARN")

log(f"Audit columns added. Output columns ({len(updated_df.columns)}): {updated_df.columns}")
log(f"Step 2 completed in {time.time() - step_start:.2f}s")

In [0]:
step_start = time.time()
log(f"Step 3: Writing to {target_table} using load_type={normalised_load_type}")

if normalised_load_type=='FULL':
    log("FULL load: overwriting target table")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
elif normalised_load_type=='APPEND':
    log("APPEND load: appending rows to target table")
    updated_df.write.mode("append").saveAsTable(target_table)
elif normalised_load_type=='MERGE':
    target_exists = spark.catalog.tableExists(target_table)
    log(f"MERGE load: target table exists = {target_exists}")

    if not target_exists:
        log("Target does not exist yet; creating it with an initial overwrite")
        updated_df.write.mode("overwrite").saveAsTable(target_table)
    else:
        source_view = 'silver_merge_source'
        updated_df.createOrReplaceTempView(source_view)

        merge_condition = " AND ".join([f"target.`{key}` = source.`{key}`" for key in merge_key_list])

        update_condition = ", ".join([f"target.`{column}`= source.`{column}`" for column in updated_df.columns if column!="insert_timestamp"])

        insert_columns = ", ".join([f"`{column}`" for column in updated_df.columns])
        insert_values = ", ".join([f"source.`{column}`" for column in updated_df.columns])

        merge_sql = f"""
        MERGE INTO {target_table} as target
        USING {source_view} as source
        ON {merge_condition}
        WHEN MATCHED THEN UPDATE SET {update_condition}
        WHEN NOT MATCHED THEN INSERT ({insert_columns}) VALUES ({insert_values})
        """.strip()
        log(f"Merge condition : {merge_condition}", "DEBUG")
        log(f"Generated MERGE SQL:\n{merge_sql}", "DEBUG")
        try:
            merge_result = spark.sql(merge_sql)
            log(f"MERGE metrics: {merge_result.collect()[0].asDict()}")
        except Exception as e:
            log(f"MERGE failed: {e}", "ERROR")
            raise
else:
    log(f"Unsupported load_type '{normalised_load_type}'", "ERROR")
    raise ValueError("load type should be MERGE, APPEND or FULL")

record_count = updated_df.count()
log(f"Rows processed in this run: {record_count}")
log(f"Rows now in {target_table}: {spark.read.table(target_table).count()}")
log(f"Step 3 completed in {time.time() - step_start:.2f}s")
log(f"========== Bronze_to_Silver finished in {time.time() - notebook_start:.2f}s ==========")

In [0]:
dbutils.notebook.exit(str(record_count))